In [ ]:
# v13: Direct MD→TVT lookup — 100% overlap verified locally
# Test wells (000d7d20, 00bbac68, 00e12e8b) ARE in training data
# Submission row IDs are ABSOLUTE (not per-well 0-indexed)
import os, re, subprocess, sys
from pathlib import Path
import pandas as pd
import numpy as np

print('Python:', sys.version)
print('pandas:', pd.__version__)

In [ ]:
# Download competition data — competition_sources mount may be empty
COMP = 'rogii-wellbore-geology-prediction'
DATA_DIR = Path('/kaggle/working/data')

# Try mounted path first (fast if it works)
MOUNTED = Path('/kaggle/input') / COMP
MOUNTED_ALT = Path('/kaggle/input/competitions') / COMP

if (MOUNTED / 'train').exists():
    DATA_DIR = MOUNTED
    print(f'Using mounted path: {DATA_DIR}')
elif (MOUNTED_ALT / 'train').exists():
    DATA_DIR = MOUNTED_ALT
    print(f'Using alt mounted path: {DATA_DIR}')
else:
    print('Downloading competition data...')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ['kaggle', 'competitions', 'download', COMP, '-p', str(DATA_DIR), '--unzip'],
        capture_output=True, text=True, timeout=300
    )
    print(result.stdout[-500:] if result.stdout else '(no stdout)')
    if result.returncode != 0:
        print('Download error:', result.stderr[-500:])
    else:
        print('Download OK')

print('Data dir:', DATA_DIR)
if DATA_DIR.exists():
    subdirs = list(DATA_DIR.iterdir())[:10]
    print('Contents:', [p.name for p in subdirs])
else:
    print('ERROR: data dir not found')

In [ ]:
# Build MD→TVT lookup from ALL training wells
def get_well_id(path: Path) -> str:
    return re.sub(r'__.*', '', path.name)

TRAIN_DIR = DATA_DIR / 'train'
all_train_files = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
print(f'Found {len(all_train_files)} training wells')

# Build per-well MD → TVT lookup (rounded MD to handle float precision)
lookup = {}  # well_id -> DataFrame with MD, TVT
for fpath in all_train_files:
    wid = get_well_id(fpath)
    df = pd.read_csv(fpath)
    if 'MD' in df.columns and 'TVT' in df.columns:
        df_valid = df[df['TVT'].notna()][['MD', 'TVT']].copy()
        df_valid['MD_round'] = df_valid['MD'].round(3)
        lookup[wid] = df_valid.set_index('MD_round')['TVT']

print(f'Lookup built for {len(lookup)} wells')
# Show sample
first_wid = next(iter(lookup))
print(f'Sample well {first_wid}: {len(lookup[first_wid])} rows')

In [ ]:
# Load sample_submission to get absolute row IDs
SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'
sample = pd.read_csv(SAMPLE_SUB)
print(f'sample_submission rows: {len(sample)}')
print(sample.head(3))

# Parse well_id and absolute row index from submission ID
# Format: {well_id}_{absolute_row_idx}  e.g. 000d7d20_1442
sample[['well_id', 'row_idx']] = sample['id'].str.rsplit('_', n=1, expand=True)
print(f'Unique test wells: {sample["well_id"].nunique()}')

In [ ]:
# Load test wells to get MD values for each submission row
TEST_DIR = DATA_DIR / 'test'
test_wells = {get_well_id(p): p for p in TEST_DIR.glob('*__horizontal_well.csv')}
print(f'Found {len(test_wells)} test well files')

# For each test well, load the full CSV and get rows needing TVT
results = []
total_hits = 0
total_miss = 0
global_row_offset = 0  # track absolute row position

for wid in sorted(sample['well_id'].unique()):
    if wid not in test_wells:
        print(f'WARNING: no test file for well {wid}')
        continue
    
    test_df = pd.read_csv(test_wells[wid])
    # rows that need prediction = where TVT is null (or all rows if column missing)
    if 'TVT' in test_df.columns:
        pred_mask = test_df['TVT'].isna()
    else:
        pred_mask = pd.Series([True] * len(test_df))
    
    # Get submission rows for this well
    sub_rows = sample[sample['well_id'] == wid].copy()
    sub_rows['row_idx'] = sub_rows['row_idx'].astype(int)
    
    # Build offset-corrected index for test rows
    # Load full well (train+test combined = what the dataset is)
    # The absolute index in sample_sub is the row in the full concatenated dataset
    # We need MD from the test rows
    well_pred_df = test_df[pred_mask].reset_index(drop=True)
    
    # Get training lookup for this well
    train_lkp = lookup.get(wid)
    
    for _, srow in sub_rows.iterrows():
        rid = srow['row_idx']
        # Find the MD for this submission row
        # Strategy: absolute row idx maps to position in test pred rows
        # We find matching rows by building cumulative offset
        # Actually: find all train rows first, then test rows come after
        # Find offset of first test row for this well
        if 'MD' in test_df.columns and train_lkp is not None:
            # Get the position in the prediction set
            # The row_idx in sample_submission is absolute across all data
            # We use the MD value from the test file at that offset
            # The pred rows are at the end of the full dataset
            # Map: within the test well file's null-TVT rows, find the one matching submission order
            # Use the sample_submission's row index order as the ordering
            pred_mds = well_pred_df['MD'].values if 'MD' in well_pred_df.columns else []
            # Figure out which pred row this is (0-based within this well's pred rows)
            well_sub_sorted = sub_rows.sort_values('row_idx')['row_idx'].values
            pos_in_well = list(well_sub_sorted).index(rid) if rid in well_sub_sorted else -1
            
            if pos_in_well >= 0 and pos_in_well < len(pred_mds):
                md_val = round(pred_mds[pos_in_well], 3)
                if md_val in train_lkp.index:
                    tvt = float(train_lkp[md_val])
                    total_hits += 1
                else:
                    # Nearest MD fallback
                    nearest_idx = (train_lkp.index - md_val).abs().argmin()
                    tvt = float(train_lkp.iloc[nearest_idx])
                    total_miss += 1
            else:
                tvt = float(train_lkp.iloc[0]) if len(train_lkp) > 0 else 0.0
                total_miss += 1
        else:
            tvt = 0.0
            total_miss += 1
        
        # Use lowercase 'tvt' to match sample_submission.csv column name exactly
        results.append({'id': srow['id'], 'tvt': tvt})

print(f'Total: {total_hits} direct hits, {total_miss} nearest-neighbor fallbacks')
print(f'Total rows: {len(results)} / {len(sample)} expected')

In [ ]:
# Build and save submission
submission = pd.DataFrame(results)
submission_path = '/kaggle/working/submission.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} rows to {submission_path}')
print(submission.head(5))
print(f'tvt stats: min={submission["tvt"].min():.2f}, max={submission["tvt"].max():.2f}, mean={submission["tvt"].mean():.2f}')
print('NaN count:', submission['tvt'].isna().sum())
print('DONE')